In [1]:
import dotenv
from langchain_openai import ChatOpenAI
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

# Access environment variables
os.environ["LANGCHAIN_API_KEY"]
# os.environ["LANGCHAIN_ENDPOINT"]
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [9]:
from typing import Optional
import requests
from bs4 import BeautifulSoup
from langchain_community.document_loaders import AsyncHtmlLoader, WebBaseLoader
from langchain_community.document_transformers import BeautifulSoupTransformer
import trafilatura
from urllib.parse import urlparse
import logging

class RobustWebScraper:
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

    def _clean_text(self, text: str) -> str:
        """Clean extracted text by removing extra whitespace and empty lines."""
        if not text:
            return ""
        lines = [line.strip() for line in text.split('\n')]
        return '\n'.join(line for line in lines if line)

    def _extract_with_trafilatura(self, url: str) -> Optional[str]:
        """Extract text using trafilatura library."""
        try:
            downloaded = trafilatura.fetch_url(url)
            if downloaded:
                text = trafilatura.extract(downloaded, include_comments=False, 
                                         include_tables=False, no_fallback=False)
                return self._clean_text(text)
        except Exception as e:
            self.logger.warning(f"Trafilatura extraction failed: {e}")
        return None

    def _extract_with_beautifulsoup(self, url: str) -> Optional[str]:
        """Extract text using BeautifulSoup."""
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Remove unwanted elements
            for element in soup(['script', 'style', 'nav', 'header', 'footer', 'aside']):
                element.decompose()
            
            # Try to find main content
            main_content = None
            for selector in ['article', 'main', '[role="main"]', '.main-content', '#main-content']:
                main_content = soup.select_one(selector)
                if main_content:
                    break
            
            text = main_content.get_text() if main_content else soup.get_text()
            return self._clean_text(text)
        except Exception as e:
            self.logger.warning(f"BeautifulSoup extraction failed: {e}")
        return None

    def _extract_with_langchain(self, url: str) -> Optional[str]:
        """Extract text using LangChain's built-in loaders."""
        try:
            # Try AsyncHtmlLoader first
            loader = AsyncHtmlLoader([url])
            docs = loader.load()
            
            if docs and docs[0].page_content.strip():
                # Transform with BS4 to clean it up
                bs_transformer = BeautifulSoupTransformer()
                docs_transformed = bs_transformer.transform_documents(docs)
                return self._clean_text(docs_transformed[0].page_content)
                
            # If AsyncHtmlLoader fails, try WebBaseLoader
            loader = WebBaseLoader(url)
            docs = loader.load()
            if docs:
                return self._clean_text(docs[0].page_content)
                
        except Exception as e:
            self.logger.warning(f"LangChain extraction failed: {e}")
        return None

    def scrape(self, url: str) -> Optional[str]:
        """
        Try multiple methods to extract text from a webpage.
        Returns the first successful extraction or None if all methods fail.
        """
        self.logger.info(f"Starting extraction from: {url}")
        
        # Validate URL
        try:
            result = urlparse(url)
            if not all([result.scheme, result.netloc]):
                raise ValueError("Invalid URL")
        except Exception as e:
            self.logger.error(f"Invalid URL: {e}")
            return None

        # Try each method in order
        methods = [
            (self._extract_with_trafilatura, "Trafilatura"),
            (self._extract_with_beautifulsoup, "BeautifulSoup"),
            (self._extract_with_langchain, "LangChain")
        ]

        for extract_method, method_name in methods:
            self.logger.info(f"Trying {method_name} method...")
            content = extract_method(url)
            if content:
                self.logger.info(f"Successfully extracted content using {method_name}")
                return content

        self.logger.error("All extraction methods failed")
        return None

In [10]:

# Example usage
if __name__ == "__main__":
    scraper = RobustWebScraper()
    url = "https://www.scientificamerican.com/article/how-the-seven-bridges-of-koenigsberg-spawned-new-math/"
    content = scraper.scrape(url)
    
    if content:
        print("Successfully extracted content:")
        print("-" * 50)
        print(content[:500] + "...")  # Print first 500 characters
    else:
        print("Failed to extract content from the webpage")

INFO: Starting extraction from: https://www.scientificamerican.com/article/how-the-seven-bridges-of-koenigsberg-spawned-new-math/
INFO: Trying Trafilatura method...
INFO: Successfully extracted content using Trafilatura


Successfully extracted content:
--------------------------------------------------
During the 18th century the denizens of the Prussian city of Königsberg wrestled with a puzzle: How could they find a walking path through the city that crossed each of its storied seven bridges exactly once?
The bridges spanned a river containing two large islands. No matter how much they strategized their routes, they couldn’t avoid repeating a bridge.
The problem stymied local thinkers, who eventually wrote a letter to famed mathematician Leonhard Euler (pronounced “oiler”) begging him to lay...


In [12]:
content

"During the 18th century the denizens of the Prussian city of Königsberg wrestled with a puzzle: How could they find a walking path through the city that crossed each of its storied seven bridges exactly once?\nThe bridges spanned a river containing two large islands. No matter how much they strategized their routes, they couldn’t avoid repeating a bridge.\nThe problem stymied local thinkers, who eventually wrote a letter to famed mathematician Leonhard Euler (pronounced “oiler”) begging him to lay their curiosity to rest. Euler responded dismissively, claiming the problem had “little relationship to mathematics.” In a way, he was right, because the relevant math hadn’t been invented yet. Despite his initial demurral, Euler did end up solving the puzzle of the seven bridges of Königsberg, unaware that in the process he had birthed two new branches of math.\nOn supporting science journalism\nIf you're enjoying this article, consider supporting our award-winning journalism by subscribing

# Summarizing

In [15]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# Example usage with ChatOpenAI
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

# Define prompt
prompt = ChatPromptTemplate.from_messages(
    [("system", "Write a concise summary of the following:\\n\\n{context}")]
)

# Instantiate chain
chain = create_stuff_documents_chain(llm, prompt)

# Wrap content in a Document object
docs = [Document(page_content=content)]

# Invoke chain
result = chain.invoke({"context": docs})
print(result)

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In the 18th century, residents of Königsberg faced a challenge: to find a walking path that crossed each of the city's seven bridges exactly once. Despite their efforts, they could not solve the problem and sought help from mathematician Leonhard Euler. Initially dismissive, Euler eventually tackled the puzzle, leading to the creation of graph theory and topology. He abstracted the problem into a graph, representing land as vertices and bridges as edges, and established conditions for the existence of Eulerian paths. His findings revealed that the Königsberg bridges could not be traversed as desired, as all four vertices had an odd number of edges. Euler's work laid the foundation for modern mathematics, demonstrating the power of abstraction in solving complex problems.


In the 18th century, residents of Königsberg faced a challenge: to find a walking path that crossed each of the city's seven bridges exactly once. Despite their efforts, they could not solve the problem and sought help from mathematician Leonhard Euler. Initially dismissive, Euler eventually tackled the puzzle, leading to the creation of graph theory and topology. He abstracted the problem into a graph, representing land as vertices and bridges as edges, and established conditions for the existence of Eulerian paths. His findings revealed that the Königsberg bridges could not be traversed as desired, as all four vertices had an odd number of edges. Euler's work laid the foundation for modern mathematics, demonstrating the power of abstraction in solving complex problems.
